In [ ]:
# ── Download cross-account datasets that don't auto-mount ─────────────────
import subprocess, os
from pathlib import Path

def _download_if_missing(dataset_slug: str, expected_path: str):
    p = Path(expected_path)
    if p.exists():
        print(f"✓ {expected_path} already mounted")
        return
    name = dataset_slug.split("/")[-1]
    # Download to /tmp (always writable; /kaggle/input is read-only for writes)
    tmp_dir = Path(f"/tmp/{name}")
    tmp_dir.mkdir(parents=True, exist_ok=True)
    print(f"⬇ Downloading {dataset_slug} → {tmp_dir} ...")
    print(f"  KAGGLE_USERNAME={os.environ.get('KAGGLE_USERNAME', 'NOT SET')}")
    r = subprocess.run(
        ["kaggle", "datasets", "download", dataset_slug,
         "-p", str(tmp_dir), "--unzip"],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        raise RuntimeError(
            f"Download failed (rc={r.returncode}) for {dataset_slug}\n"
            f"STDOUT: {r.stdout[-800:]}\nSTDERR: {r.stderr[-800:]}"
        )
    print(f"✓ Downloaded to {tmp_dir}")
    print(r.stdout[-200:] if r.stdout else "")
    # Detect content root (zip may add a top-level folder)
    contents = list(tmp_dir.iterdir())
    content_root = contents[0] if len(contents) == 1 and contents[0].is_dir() else tmp_dir
    print(f"  Content root: {content_root}")
    print(f"  Contents: {[x.name for x in list(content_root.iterdir())[:8]]}")
    # Create symlink at expected path so downstream code finds it there
    try:
        p.parent.mkdir(parents=True, exist_ok=True)
        if not p.exists():
            os.symlink(str(content_root), str(p))
            print(f"✓ Symlinked {content_root} → {p}")
    except OSError as e:
        print(f"⚠ Could not symlink at {p}: {e}")
        print(f"  Weights available at: {content_root}")

_download_if_missing("gumfreddy/mtmc-weights", "/kaggle/input/mtmc-weights")

# Notebook 09q: Vehicle ReID -- CityFlowV2 Extended Fine-Tuning from the 80.14% mAP TransReID Checkpoint
**Multi-Camera Tracking System -- Kaggle Training Pipeline**

Resume the original TransReID ViT-Base/16 CLIP CityFlowV2 recipe from the best 09 checkpoint and continue fine-tuning for **120 more epochs** with a lower learning rate. This keeps the same augmentation stack, the same TransReID architecture, and the same CityFlowV2 crop-generation pipeline while rebuilding the optimizer and cosine schedule from scratch.

## Why this extension?
- **Warm start from the best primary model**: resume from the original 80.14% mAP CityFlowV2 checkpoint
- **Lower-LR continuation**: halve both backbone and head learning rates for a gentler fine-tuning phase
- **Same proven architecture**: TransReID ViT-Base/16 CLIP with SIE, JPM, and BNNeck
- **Same data recipe**: identical CityFlowV2 download, crop extraction, and train/query/gallery splits
- **Safer evaluation**: chunked metric computation to avoid full distance-matrix OOM

## Strategy
1. Download CityFlowV2 from Google Drive and flatten the train/validation camera directories
2. Extract crops from GT annotations and video frames using the original 09 logic
3. Build the same train/query/gallery split protocol as the original 09 run
4. Recreate the TransReID ViT-Base/16 CLIP + SIE + JPM + BNNeck model exactly
5. Load the full state dict from `/kaggle/input/mtmc-weights/models/reid/transreid_cityflowv2_best.pth`
6. Rebuild optimizer and cosine schedule from scratch, then continue training for 120 epochs with lower LR
7. Evaluate with chunked distance computation and save the best extended checkpoint as `transreid_cityflowv2_extended_best.pth`

## Model
| Model | Architecture | Dim | Init | Target |
|-------|-------------|-----|------|--------|
| **TransReID Extended** | ViT-Base/16 CLIP + SIE + JPM | 768 | 09 best checkpoint | >80% mAP |

**Runtime**: GPU T4 x2 (32GB) | **Duration**: ~3-4h (120 epochs, resumed fine-tune, 59 cameras)

In [ ]:
!pip install -q timm matplotlib pandas loguru gdown

## 1. Setup

In [ ]:
import os
import sys
import json
import time
import re
import shutil
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from torch.optim import Adam, SGD
import torchvision.transforms as T
from PIL import Image
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
NUM_GPUS = 1  # Force single-GPU: DataParallel replication fails with timm ViT patch_embed
print(f"GPUs available: {NUM_GPUS}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({props.total_memory / 1024**3:.1f} GB)")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = Path("/kaggle/working/vehicle_reid_cityflowv2_extended")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR = Path("/kaggle/working/exported_models")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

RESUME_CHECKPOINT_CANDIDATES = [
    Path("/kaggle/input/09-vehicle-reid-cityflowv2-circleloss-ablation/vehicle_reid_cityflowv2/transreid_cityflowv2_best.pth"),
    Path("/kaggle/input/mtmc-weights/reid/transreid_cityflowv2_best.pth"),
    Path("/kaggle/input/mtmc-weights/models/reid/transreid_cityflowv2_best.pth"),
    Path("/kaggle/input/mtmc-weights/transreid_cityflowv2_best.pth"),
    # Correct filename from gumfreddy/mtmc-weights dataset
    Path("/kaggle/input/mtmc-weights/vehicle_transreid_vit_base_cityflowv2.pth"),
    # Fallback: if /tmp download was used and symlink failed
    Path("/tmp/mtmc-weights/models/reid/transreid_cityflowv2_best.pth"),
    Path("/tmp/mtmc-weights/reid/transreid_cityflowv2_best.pth"),
    Path("/tmp/mtmc-weights/transreid_cityflowv2_best.pth"),
    Path("/tmp/mtmc-weights/vehicle_transreid_vit_base_cityflowv2.pth"),
]


def debug_list_top_level(root: Path):
    if not root.exists():
        return ["<missing>"]
    entries = sorted(path.name for path in root.iterdir())
    if not entries:
        return ["<empty>"]
    return entries[:20]


RESUME_CHECKPOINT_PATH = next((path for path in RESUME_CHECKPOINT_CANDIDATES if path.exists()), None)
if RESUME_CHECKPOINT_PATH is None:
    debug_payload = {
        "09_kernel_root": debug_list_top_level(Path("/kaggle/input/09-vehicle-reid-cityflowv2-circleloss-ablation")),
        "mtmc_weights_root": debug_list_top_level(Path("/kaggle/input/mtmc-weights")),
    }
    raise FileNotFoundError(
        "Missing resume checkpoint; checked candidates: "
        + ", ".join(str(path) for path in RESUME_CHECKPOINT_CANDIDATES)
        + ". Debug roots: "
        + json.dumps(debug_payload, ensure_ascii=True)
    )

BEST_MODEL_PATH = OUTPUT_DIR / "transreid_cityflowv2_extended_best.pth"
BEST_MODEL_PATH_A = OUTPUT_DIR / "transreid_cityflowv2_extended_A_best.pth"  # Experiment A (from checkpoint)
BEST_MODEL_PATH_B = OUTPUT_DIR / "transreid_cityflowv2_extended_B_best.pth"  # Experiment B (from scratch)
LAST_MODEL_PATH = OUTPUT_DIR / "transreid_cityflowv2_extended_last.pth"
EVAL_CHUNK_SIZE = 1024

print(f"\nDevice: {DEVICE} | GPUs: {NUM_GPUS}")
print("Resume checkpoint candidates:")
for candidate in RESUME_CHECKPOINT_CANDIDATES:
    print(f"  - {candidate}")
print(f"Resolved resume checkpoint: {RESUME_CHECKPOINT_PATH}")

## 1.5 Download CityFlowV2 from Google Drive

CityFlowV2 (AI City Challenge 2022 Track 1) is not on Kaggle Datasets.
We download it from Google Drive using `gdown`, then extract and reorganize
the camera directories.

In [ ]:
# -- Download CityFlowV2 from Google Drive --
# All large files go to /tmp/ to avoid filling /kaggle/working/ (~20GB limit).
import subprocess, zipfile, re as _re

CITYFLOW_DIR = Path("/tmp/cityflowv2")
GDRIVE_ID = "13wNJpS_Oaoe-7y5Dzexg_Ol7bKu1OWuC"
ARCHIVE_NAME = "AIC22_Track1_MTMC_Tracking.zip"

# CityFlowV2 is an MTMC benchmark: vehicle IDs in gt.txt are GLOBALLY consistent
# across all 46 cameras / all splits. 880 distinct vehicle identities total.
# We use train + validation (have GT); test (S06) has no GT.
ALLOWED_SPLITS = {"train", "validation"}

# Check if already available (e.g. from previous run or attached dataset)
already_found = False
for check_dir in [CITYFLOW_DIR, Path("/kaggle/input/cityflowv2"), Path("/kaggle/input/aic22-track1-mtmc-tracking")]:
    if check_dir.exists() and any(list(check_dir.rglob("vdo.avi"))[:1]):
        print(f"CityFlowV2 already available at {check_dir}")
        CITYFLOW_DIR = check_dir
        already_found = True
        break

if not already_found:
    CITYFLOW_DIR.mkdir(parents=True, exist_ok=True)
    archive_path = Path(f"/tmp/{ARCHIVE_NAME}")

    if not archive_path.exists():
        print(f"Downloading CityFlowV2 from Google Drive (id={GDRIVE_ID})...")
        print("This may take 10-20 minutes for the full dataset (~20GB).")
        import gdown
        gdown.download(
            f"https://drive.google.com/uc?id={GDRIVE_ID}",
            str(archive_path), quiet=False
        )
    else:
        print(f"Archive already downloaded: {archive_path}")

    # Extract to a staging dir to avoid polluting /tmp/
    staging = Path("/tmp/_aic22_staging")
    staging.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {archive_path} to {staging}...")
    with zipfile.ZipFile(str(archive_path), "r") as zf:
        zf.extractall(str(staging))

    # Delete archive immediately to reclaim ~20GB
    if archive_path.exists():
        archive_path.unlink()
        print("Deleted archive to reclaim space.")

    # Debug: show what was extracted
    print("\nExtracted contents of staging dir:")
    for item in sorted(staging.iterdir()):
        print(f"  {item.name} ({'dir' if item.is_dir() else 'file'})")

    # Find all vdo.avi files under staging and flatten into CITYFLOW_DIR
    # Keep train + validation (both have GT); skip test (S06, no GT).
    moved = 0
    skipped_splits = set()
    for vdo_path in sorted(staging.rglob("vdo.avi")):
        cam_dir = vdo_path.parent       # e.g. .../c001/
        scene_dir = cam_dir.parent      # e.g. .../S01/
        split_dir = scene_dir.parent    # e.g. .../train/
        cam_name = cam_dir.name         # e.g. c001
        scene_name = scene_dir.name     # e.g. S01
        split_name = split_dir.name     # e.g. train
        # Only use dirs that look like camera/scene names
        if not cam_name.startswith("c") or not scene_name.startswith("S"):
            print(f"  SKIP unusual path: {vdo_path}")
            continue
        if split_name not in ALLOWED_SPLITS:
            skipped_splits.add(split_name)
            print(f"  SKIP {scene_name}_{cam_name}: {split_name} split (no GT)")
            continue
        flat_name = f"{scene_name}_{cam_name}"  # e.g. S01_c001
        target = CITYFLOW_DIR / flat_name
        if not target.exists():
            shutil.move(str(cam_dir), str(target))
            moved += 1
            if moved <= 15:
                print(f"  {flat_name} -> {target}")
            elif moved == 16:
                print(f"  ... (showing first 15 only)")
    print(f"Total cameras moved: {moved}")
    if skipped_splits:
        print(f"Skipped splits: {sorted(skipped_splits)} (no GT annotations)")

    # Also check if the zip extracted directly into /tmp/ (no wrapper dir)
    if moved == 0:
        print("Trying fallback: zip may have extracted directly to /tmp/...")
        for split_name in sorted(ALLOWED_SPLITS):
            split_dir = Path(f"/tmp/{split_name}")
            if not split_dir.exists():
                continue
            for vdo_path in sorted(split_dir.rglob("vdo.avi")):
                cam_dir = vdo_path.parent
                scene_dir = cam_dir.parent
                cam_name = cam_dir.name
                scene_name = scene_dir.name
                if not cam_name.startswith("c") or not scene_name.startswith("S"):
                    continue
                flat_name = f"{scene_name}_{cam_name}"
                target = CITYFLOW_DIR / flat_name
                if not target.exists():
                    shutil.move(str(cam_dir), str(target))
                    moved += 1
                    if moved <= 15:
                        print(f"  {flat_name} -> {target}")
        print(f"Fallback cameras moved: {moved}")

    # Cleanup staging + leftover split dirs
    if staging.exists():
        shutil.rmtree(str(staging), ignore_errors=True)
    for d in [Path("/tmp/train"), Path("/tmp/test"), Path("/tmp/validation")]:
        if d.exists():
            shutil.rmtree(str(d), ignore_errors=True)
    # Clean other extracted files
    for f in [Path("/tmp/ReadMe.txt"), Path("/tmp/list_cam.txt")]:
        if f.exists():
            f.unlink()
    for f in Path("/tmp").glob("Dataset License*"):
        f.unlink(missing_ok=True)
    for d in [Path("/tmp/cam_framenum"), Path("/tmp/cam_loc"), Path("/tmp/cam_timestamp"), Path("/tmp/eval")]:
        if d.exists():
            shutil.rmtree(str(d), ignore_errors=True)

# Verify -- only count dirs matching SXX_cXXX pattern
_cam_re = _re.compile(r"^S\d{2}_c\d{3}$")
found_cams = sorted([d.name for d in CITYFLOW_DIR.iterdir() if d.is_dir() and _cam_re.match(d.name)]) if CITYFLOW_DIR.exists() else []
print(f"\nCityFlowV2 at {CITYFLOW_DIR}")
print(f"Camera dirs found: {len(found_cams)} (46 unique physical cameras)")
print(f"  {found_cams}")
if not found_cams:
    print("\nERROR: No cameras found! Listing /tmp/ for debugging:")
    for p in sorted(Path("/tmp").iterdir()):
        if not p.name.startswith("uv-") and not p.name.startswith("tmp"):
            print(f"  {p.name} ({'dir' if p.is_dir() else 'file'})")

## 2. CityFlowV2 Dataset -- Crop Extraction

CityFlowV2 provides per-camera videos + MOT-format ground truth (gt.txt).
We extract vehicle bounding-box crops from video frames, then split into
train/query/gallery for ReID training.

**Using all 59 scene-camera combinations** from train (S01,S03,S04) + validation
(S02,S05) splits. Vehicle IDs are globally consistent across all cameras --
this is a MTMC benchmark with 880 distinct vehicle identities. Only S06 (test,
no GT) is excluded.

In [ ]:
# -- Locate CityFlowV2 data --
# Use ALL cameras with GT (no target subset filtering)
TARGET_CAMS = set()  # empty = use all available cameras
MAX_CROPS_PER_ID_CAM = 20  # more crops per ID for larger dataset
MIN_AREA = 2000
MIN_BBOX_SIDE = 30
TRAIN_RATIO = 0.7
MIN_CAMS_FOR_EVAL = 2
SEED = 42

# DATA_ROOT is directory containing S0X_c0XX/ subdirs
possible_roots = [
    CITYFLOW_DIR,
    Path("/tmp/cityflowv2"),
    Path("/kaggle/input/cityflowv2"),
    Path("/kaggle/input/aic22-track1-mtmc-tracking"),
    Path("/tmp/data/raw/cityflowv2"),
    Path("data/raw/cityflowv2"),
]

DATA_ROOT = None
for root in possible_roots:
    if root.exists():
        # Check for any camera dir pattern S0X_c0XX
        cam_dirs = [d for d in root.iterdir() if d.is_dir() and "_c0" in d.name]
        if cam_dirs:
            DATA_ROOT = root
            break
    if DATA_ROOT:
        break

if DATA_ROOT is None:
    # Search more broadly
    for base in [Path("/tmp"), Path("/kaggle/input")]:
        if not base.exists():
            continue
        vdos = list(base.rglob("vdo.avi"))[:1]
        if vdos:
            # Go up to the parent containing S0X dirs
            cam = vdos[0].parent
            DATA_ROOT = cam.parent.parent if cam.parent.name.startswith("S0") else cam.parent
            break

if DATA_ROOT is None:
    raise RuntimeError(
        "CityFlowV2 not found. Either:\n"
        "  1. Attach a CityFlowV2 dataset on Kaggle\n"
        "  2. Check the download cell output for errors\n"
        "  3. Locally: python scripts/download_datasets.py --dataset cityflowv2"
    )

print(f"CityFlowV2 data root: {DATA_ROOT}")

# Find available cameras with GT + video
def find_gt(cam_dir):
    """Find gt.txt -- could be at gt.txt or gt/gt.txt"""
    direct = cam_dir / "gt.txt"
    if direct.exists():
        return direct
    nested = cam_dir / "gt" / "gt.txt"
    if nested.exists():
        return nested
    # Search for any gt.txt
    matches = list(cam_dir.rglob("gt.txt"))
    return matches[0] if matches else None

def find_video(cam_dir):
    """Find video file"""
    for ext in ["avi", "mp4"]:
        for name in ["vdo", "video"]:
            p = cam_dir / f"{name}.{ext}"
            if p.exists():
                return p
    vids = list(cam_dir.glob("*.avi")) + list(cam_dir.glob("*.mp4"))
    return vids[0] if vids else None

CAMERAS = []
cam_gt_paths = {}
cam_vid_paths = {}

for cam_dir in sorted(DATA_ROOT.iterdir()):
    if not cam_dir.is_dir():
        continue
    gt = find_gt(cam_dir)
    vid = find_video(cam_dir)
    cam_name = cam_dir.name
    if gt and vid:
        CAMERAS.append(cam_name)
        cam_gt_paths[cam_name] = gt
        cam_vid_paths[cam_name] = vid
        print(f"  OK {cam_name}: GT={gt.name} Video={vid.name}")
    else:
        missing = []
        if not gt: missing.append("GT")
        if not vid: missing.append("video")
        print(f"  SKIP {cam_name}: missing {', '.join(missing)}")

if not CAMERAS:
    raise RuntimeError("No cameras with both GT and video found!")

print(f"\nUsing {len(CAMERAS)}/{len(found_cams)} cameras: {CAMERAS}")

In [ ]:
# -- Extract vehicle crops from GT + video --
# Crops go to /tmp/ to preserve /kaggle/working/ space
CROP_DIR = Path("/tmp/cityflowv2_crops")
CROP_DIR.mkdir(parents=True, exist_ok=True)


def load_gt(gt_path):
    rows = []
    with open(gt_path) as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) < 6:
                continue
            frame, tid = int(parts[0]), int(parts[1])
            x, y, w, h = int(parts[2]), int(parts[3]), int(parts[4]), int(parts[5])
            rows.append((frame, tid, x, y, w, h))
    return rows


def extract_crops_from_camera(cam_name, gt_file, vid_file, crop_dir, max_crops, min_area):
    """Extract crops using pre-resolved GT and video file paths."""
    gt = load_gt(str(gt_file))
    if not gt:
        print(f"  {cam_name}: empty GT file")
        return {}

    id_dets = defaultdict(list)
    for frame, tid, x, y, w, h in gt:
        id_dets[tid].append((frame, x, y, w, h))

    frame_to_dets = defaultdict(list)
    for tid, dets in id_dets.items():
        if len(dets) <= max_crops:
            sampled = dets
        else:
            step = len(dets) / max_crops
            sampled = [dets[int(i * step)] for i in range(max_crops)]
        for frame, x, y, w, h in sampled:
            if w * h >= min_area and w >= MIN_BBOX_SIDE and h >= MIN_BBOX_SIDE:
                frame_to_dets[frame].append((tid, x, y, w, h))

    if not frame_to_dets:
        return {}

    crops = defaultdict(list)
    cap = cv2.VideoCapture(str(vid_file))
    current_frame = 0
    target_frames = sorted(frame_to_dets.keys())
    target_idx = 0

    while target_idx < len(target_frames):
        ret, img = cap.read()
        if not ret:
            break
        current_frame += 1
        if current_frame < target_frames[target_idx]:
            continue
        if current_frame > target_frames[target_idx]:
            while target_idx < len(target_frames) and target_frames[target_idx] < current_frame:
                target_idx += 1
            if target_idx >= len(target_frames) or current_frame != target_frames[target_idx]:
                continue

        H, W = img.shape[:2]
        for tid, x, y, w, h in frame_to_dets[current_frame]:
            x1, y1 = max(0, x), max(0, y)
            x2, y2 = min(W, x + w), min(H, y + h)
            if x2 - x1 < MIN_BBOX_SIDE or y2 - y1 < MIN_BBOX_SIDE:
                continue
            crop = img[y1:y2, x1:x2]
            fname = f"{tid:04d}_{cam_name}_f{current_frame:06d}.jpg"
            fpath = str(crop_dir / fname)
            cv2.imwrite(fpath, crop)
            crops[tid].append(fpath)
        target_idx += 1

    cap.release()
    n = sum(len(v) for v in crops.values())
    print(f"  {cam_name}: {n} crops from {len(crops)} vehicles")
    return dict(crops)


# Check if crops already exist
existing_crops = list(CROP_DIR.glob("*.jpg"))
if len(existing_crops) > 100:
    print(f"Reusing {len(existing_crops)} existing crops from {CROP_DIR}")
    all_crops = defaultdict(lambda: defaultdict(list))
    for fpath in sorted(existing_crops):
        parts = fpath.stem.split("_")
        tid = int(parts[0])
        cam = parts[1] + "_" + parts[2]
        all_crops[tid][cam].append(str(fpath))
    all_crops = {k: dict(v) for k, v in all_crops.items()}
else:
    print("Extracting crops from videos...")
    all_crops = defaultdict(lambda: defaultdict(list))
    for cam in CAMERAS:
        gt_file = cam_gt_paths[cam]
        vid_file = cam_vid_paths[cam]
        cam_crops = extract_crops_from_camera(cam, gt_file, vid_file, CROP_DIR, MAX_CROPS_PER_ID_CAM, MIN_AREA)
        for tid, paths in cam_crops.items():
            all_crops[tid][cam].extend(paths)
    all_crops = {k: dict(v) for k, v in all_crops.items()}

# After extracting crops, delete the raw video/GT data to free space
print("Cleaning up raw CityFlowV2 data to free disk space...")
if CITYFLOW_DIR.exists() and str(CITYFLOW_DIR).startswith("/tmp/"):
    shutil.rmtree(str(CITYFLOW_DIR), ignore_errors=True)
    print(f"  Removed {CITYFLOW_DIR}")

total = sum(sum(len(v) for v in cams.values()) for cams in all_crops.values())
n_ids = len(all_crops)
multi_cam = sum(1 for cams in all_crops.values() if len(cams) >= MIN_CAMS_FOR_EVAL)
print(f"\nTotal: {total} crops, {n_ids} vehicle IDs ({multi_cam} with >={MIN_CAMS_FOR_EVAL} cameras)")

# Sanity check: verify IDs are globally consistent (MTMC benchmark property)
# Vehicles seen across DIFFERENT scenes confirms global ID consistency
scene_ids = defaultdict(set)
for tid, cams in all_crops.items():
    for cam_name in cams:
        scene = cam_name.split("_")[0]  # S01, S02, etc.
        scene_ids[scene].add(tid)
cross_scene = 0
for tid, cams in all_crops.items():
    scenes = {c.split("_")[0] for c in cams}
    if len(scenes) > 1:
        cross_scene += 1
print(f"\nID sanity check:")
for s in sorted(scene_ids):
    print(f"  {s}: {len(scene_ids[s])} vehicle IDs")
print(f"  Cross-scene vehicles (same ID in 2+ scenes): {cross_scene}")
if n_ids > 880:
    print(f"  WARNING: {n_ids} IDs found but CityFlowV2 has 880 annotated. Possible ID collision!")
else:
    print(f"  OK: {n_ids} IDs <= 880 official annotated identities")

In [ ]:
# -- Build train / query / gallery splits --
if not all_crops:
    raise RuntimeError(
        f"No crops extracted! CAMERAS={CAMERAS}, CROP_DIR={CROP_DIR}\n"
        "The download or extraction likely failed. Check the download cell output."
    )

rng = np.random.RandomState(SEED)

multi_cam_ids = sorted(tid for tid, cams in all_crops.items() if len(cams) >= MIN_CAMS_FOR_EVAL)
single_cam_ids = sorted(tid for tid, cams in all_crops.items() if len(cams) < MIN_CAMS_FOR_EVAL)

# If no multi-cam IDs, relax to single-cam training (evaluation won't be cross-camera)
if not multi_cam_ids:
    print(f"WARNING: No vehicles seen in >={MIN_CAMS_FOR_EVAL} cameras.")
    print(f"  Using single-cam IDs for training (no cross-camera eval possible).")
    all_ids = sorted(all_crops.keys())
    rng.shuffle(all_ids)
    n_train = max(int(len(all_ids) * TRAIN_RATIO), 1)
    multi_cam_ids = all_ids  # treat all as "multi" for splitting
    single_cam_ids = []

rng.shuffle(multi_cam_ids)
n_train = int(len(multi_cam_ids) * TRAIN_RATIO)
train_ids = set(multi_cam_ids[:n_train])
eval_ids = set(multi_cam_ids[n_train:])

cam_names = sorted({cam for cams in all_crops.values() for cam in cams})
cam2id = {c: i for i, c in enumerate(cam_names)}

train_data, query_data, gallery_data = [], [], []

train_pid_set = sorted(train_ids)
pid2label = {tid: i for i, tid in enumerate(train_pid_set)}

for tid in sorted(train_ids):
    label = pid2label[tid]
    for cam_name, paths in all_crops[tid].items():
        camid = cam2id[cam_name]
        for p in paths:
            train_data.append((p, label, camid))

eval_pid2label = {tid: i for i, tid in enumerate(sorted(eval_ids))}
for tid in sorted(eval_ids):
    pid = eval_pid2label[tid]
    for cam_name, paths in all_crops[tid].items():
        if not paths:
            continue
        camid = cam2id[cam_name]
        idx = rng.randint(0, len(paths))
        query_data.append((paths[idx], pid, camid))
        for i, p in enumerate(paths):
            if i != idx:
                gallery_data.append((p, pid, camid))

distractor_pid = len(eval_ids)
for tid in sorted(single_cam_ids):
    for cam_name, paths in all_crops[tid].items():
        camid = cam2id[cam_name]
        for p in paths:
            gallery_data.append((p, distractor_pid, camid))
    distractor_pid += 1

num_classes = max(len(set(pid for _, pid, _ in train_data)), 1)
num_cameras = max(len(cam_names), 1)

CAN_EVAL = len(query_data) > 0 and len(gallery_data) > 0
print(f"Train:   {len(train_data)} images, {num_classes} IDs, {num_cameras} cameras")
print(f"Query:   {len(query_data)} images, {len(eval_ids)} IDs")
print(f"Gallery: {len(gallery_data)} images")
print(f"Cameras: {cam_names}")
print(f"Evaluation possible: {CAN_EVAL}")

## 3. Data Loading + Losses

In [ ]:
# -- 3. Transforms, Dataset, Samplers, DataLoaders --
# Creates train/query/gallery dataloaders from crops built above.
CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
CLIP_STD  = [0.26862954, 0.26130258, 0.27577711]
H, W = 256, 256  # resolution for TransReID 256x256

train_tf = T.Compose([
    T.Resize((H + 16, W + 16), interpolation=T.InterpolationMode.BICUBIC),
    T.RandomHorizontalFlip(p=0.5),
    T.Pad(10),
    T.RandomCrop((H, W)),
    T.ColorJitter(brightness=0.2, contrast=0.15, saturation=0.1, hue=0.0),
    T.ToTensor(),
    T.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
    T.RandomErasing(p=0.5, scale=(0.02, 0.33), ratio=(0.3, 3.3), value="random"),
])

test_tf = T.Compose([
    T.Resize((H, W), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
])


class ReIDDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        path, pid, cam = self.data[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, pid, cam, path


class PKSampler(Sampler):
    def __init__(self, data_source, p=16, k=4):
        self.data_source = data_source
        self.p = p
        self.k = k
        self.pid_to_indices = defaultdict(list)
        for idx, (_, pid, _) in enumerate(data_source):
            self.pid_to_indices[pid].append(idx)
        self.pids = list(self.pid_to_indices.keys())
        self.length = max(len(self.pids), 1) * self.k

    def __iter__(self):
        order = np.random.permutation(self.pids)
        batch = []
        for pid in order:
            idxs = self.pid_to_indices[pid]
            replace = len(idxs) < self.k
            choice = np.random.choice(idxs, self.k, replace=replace)
            batch.extend(choice.tolist())
        return iter(batch)

    def __len__(self):
        return self.length


train_ds = ReIDDataset(train_data, train_tf)
query_ds = ReIDDataset(query_data, test_tf)
gallery_ds = ReIDDataset(gallery_data, test_tf)

train_loader = DataLoader(
    train_ds,
    batch_size=64,
    sampler=PKSampler(train_data, p=16, k=4),
    num_workers=4,
    pin_memory=True,
    drop_last=True,
)
query_loader = DataLoader(query_ds, batch_size=64, num_workers=4, pin_memory=True)
gallery_loader = DataLoader(gallery_ds, batch_size=64, num_workers=4, pin_memory=True)

print(f"Train batches: {len(train_loader)}")
print(f"Query batches: {len(query_loader)}")
print(f"Gallery batches: {len(gallery_loader)}")

In [ ]:
@torch.no_grad()
def extract_features(model, loader, device="cuda", flip=True, pass_cams=False):
    model.eval()
    feats, pids, cams = [], [], []
    for imgs, pid, cam, _ in loader:
        imgs = imgs.to(device)
        kwargs = {"cam_ids": cam.to(device).long()} if pass_cams else {}
        f = model(imgs, **kwargs)
        if isinstance(f, (tuple, list)):
            f = f[-1]
        if flip:
            ff = model(torch.flip(imgs, [3]), **kwargs)
            if isinstance(ff, (tuple, list)):
                ff = ff[-1]
            f = (f + ff) / 2
        f = F.normalize(f, p=2, dim=1)
        feats.append(f.cpu().numpy())
        pids.append(pid.numpy())
        cams.append(cam.numpy())
    if not feats:
        return np.zeros((0, 768), dtype=np.float32), np.zeros(0, dtype=int), np.zeros(0, dtype=int)
    return np.concatenate(feats), np.concatenate(pids), np.concatenate(cams)


def eval_market1501_chunked(query_features, gallery_features, q_pids, g_pids, q_cams, g_cams, chunk_size=1024, max_rank=50):
    if query_features.shape[0] == 0 or gallery_features.shape[0] == 0:
        return 0.0, np.zeros(max_rank, dtype=np.float32)

    num_q = query_features.shape[0]
    num_g = gallery_features.shape[0]
    max_rank = min(max_rank, num_g)
    gallery_features_t = gallery_features.T

    all_cmc = []
    all_ap = []

    for start in range(0, num_q, chunk_size):
        end = min(start + chunk_size, num_q)
        q_chunk = query_features[start:end]
        q_pids_chunk = q_pids[start:end]
        q_cams_chunk = q_cams[start:end]

        dist_chunk = 1.0 - q_chunk @ gallery_features_t
        indices = np.argsort(dist_chunk, axis=1)
        matches = (g_pids[indices] == q_pids_chunk[:, None]).astype(np.int32)

        for local_idx in range(end - start):
            q_pid = q_pids_chunk[local_idx]
            q_cam = q_cams_chunk[local_idx]
            order = indices[local_idx]
            remove = (g_pids[order] == q_pid) & (g_cams[order] == q_cam)
            keep = ~remove
            raw_cmc = matches[local_idx][keep]
            if raw_cmc.sum() == 0:
                continue

            cmc = raw_cmc.cumsum()
            cmc[cmc > 1] = 1
            cmc = cmc[:max_rank]
            if cmc.shape[0] < max_rank:
                pad_value = int(cmc[-1]) if cmc.shape[0] else 0
                cmc = np.pad(cmc, (0, max_rank - cmc.shape[0]), constant_values=pad_value)
            all_cmc.append(cmc)

            num_rel = raw_cmc.sum()
            tmp_cmc = raw_cmc.cumsum()
            precision = tmp_cmc / (np.arange(len(tmp_cmc)) + 1.0)
            all_ap.append(float((precision * raw_cmc).sum() / num_rel))

        del q_chunk, q_pids_chunk, q_cams_chunk, dist_chunk, indices, matches

    if not all_cmc:
        return 0.0, np.zeros(max_rank, dtype=np.float32)

    return float(np.mean(all_ap)), np.asarray(all_cmc, dtype=np.float32).mean(axis=0)


def full_eval(model, ql, gl, device="cuda", rerank=False, pass_cams=False):
    qf, qp, qc = extract_features(model, ql, device, pass_cams=pass_cams)
    gf, gp, gc = extract_features(model, gl, device, pass_cams=pass_cams)
    if qf.shape[0] == 0 or gf.shape[0] == 0:
        print("  [WARN] Empty query or gallery -- skipping eval")
        return 0.0, np.zeros(50, dtype=np.float32), None, None

    mAP, cmc = eval_market1501_chunked(
        qf,
        gf,
        qp,
        gp,
        qc,
        gc,
        chunk_size=EVAL_CHUNK_SIZE,
    )
    return mAP, cmc, None, None

print(f"Chunked evaluation ready (chunk_size={EVAL_CHUNK_SIZE})")

## 5. TransReID Model

Same architecture as the original 09 notebook: ViT-Base/16 CLIP + SIE + JPM + BNNeck routing fix.

**Key difference**: we recreate the identical TransReID architecture, then load the full CityFlowV2-trained 09 checkpoint instead of starting from VeRi-776 or raw CLIP weights. This keeps:
- Vehicle-specific CityFlowV2 feature representations already learned by the best primary model
- The same camera-aware SIE formulation for CityFlowV2 camera topology
- The same JPM patch-mixing branch and BNNeck classifier routing

In [ ]:
import copy
import timm

VIT_MODEL = "vit_base_patch16_clip_224.openai"
IMG_SIZE = (256, 256)
H, W = IMG_SIZE
print(f"Backbone: {VIT_MODEL}")


class TransReID(nn.Module):
    def __init__(self, num_classes, num_cameras=0, embed_dim=768,
                 vit_model="vit_base_patch16_clip_224.openai", pretrained=True,
                 sie_camera=True, jpm=True, img_size=224):
        super().__init__()
        self.sie_camera = sie_camera and num_cameras > 0
        self.jpm = jpm
        self.vit = timm.create_model(vit_model, pretrained=pretrained,
                                     num_classes=0, img_size=img_size)
        self.vit_dim = self.vit.embed_dim

        self.num_blocks = len(self.vit.blocks)

        if self.sie_camera:
            self.sie_embed = nn.Parameter(torch.zeros(num_cameras, 1, self.vit_dim))
            nn.init.trunc_normal_(self.sie_embed, std=0.02)

        self.bn = nn.BatchNorm1d(self.vit_dim)
        self.bn.bias.requires_grad_(False)

        self.proj = nn.Linear(self.vit_dim, embed_dim, bias=False) if embed_dim != self.vit_dim else nn.Identity()
        self.cls_head = nn.Linear(embed_dim, num_classes, bias=False)
        if isinstance(self.proj, nn.Linear):
            nn.init.kaiming_normal_(self.proj.weight, mode="fan_out")
        nn.init.normal_(self.cls_head.weight, std=0.001)

        if self.jpm:
            self.bn_jpm = nn.BatchNorm1d(self.vit_dim)
            self.bn_jpm.bias.requires_grad_(False)
            self.jpm_cls = nn.Linear(self.vit_dim, num_classes, bias=False)
            nn.init.normal_(self.jpm_cls.weight, std=0.001)

        print(f"TransReID: {vit_model}, dim={self.vit_dim}, embed={embed_dim}, "
              f"SIE={self.sie_camera}({num_cameras}), JPM={jpm}, img={img_size}")

    def forward(self, x, cam_ids=None):
        B = x.shape[0]
        x = self.vit.patch_embed(x)
        if hasattr(self.vit, '_pos_embed'):
            x = self.vit._pos_embed(x)
        else:
            cls_tok = self.vit.cls_token.expand(B, -1, -1)
            x = torch.cat([cls_tok, x], dim=1) + self.vit.pos_embed
            if hasattr(self.vit, 'pos_drop'):
                x = self.vit.pos_drop(x)

        if self.sie_camera and cam_ids is not None:
            x = x + self.sie_embed[cam_ids]

        if hasattr(self.vit, 'patch_drop'):
            x = self.vit.patch_drop(x)
        if hasattr(self.vit, 'norm_pre'):
            x = self.vit.norm_pre(x)

        for blk in self.vit.blocks:
            x = blk(x)
        x = self.vit.norm(x)

        g = x[:, 0]
        bn = self.bn(g)
        proj = self.proj(bn)

        if self.training:
            cls = self.cls_head(proj)
            if self.jpm:
                patches = x[:, 1:]
                idx = torch.randperm(patches.size(1), device=x.device)
                shuffled = patches[:, idx]
                mid = shuffled.size(1) // 2
                jpm_feat = (shuffled[:, :mid].mean(1) + shuffled[:, mid:].mean(1)) / 2
                return cls, g, self.jpm_cls(self.bn_jpm(jpm_feat))
            return cls, g
        return F.normalize(proj, p=2, dim=1)

    def get_llrd_param_groups(self, backbone_lr, head_lr, decay=0.75):
        groups = {}
        for name, param in self.named_parameters():
            if not param.requires_grad:
                continue
            if name.startswith("vit."):
                if "blocks." in name:
                    block_idx = int(name.split("blocks.")[1].split(".")[0])
                    depth = block_idx + 1
                elif any(key in name for key in ["patch_embed", "cls_token", "pos_embed", "norm_pre"]):
                    depth = 0
                else:
                    depth = self.num_blocks + 1
                scale = decay ** (self.num_blocks + 1 - depth)
                lr = backbone_lr * scale
                group_key = f"bb_d{depth}"
            else:
                lr = head_lr
                group_key = "head"
            groups.setdefault(group_key, {"params": [], "lr": lr})["params"].append(param)
        return sorted(groups.values(), key=lambda group: group["lr"])


class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.ema = copy.deepcopy(model)
        self.ema.eval()
        self.decay = decay
        for param in self.ema.parameters():
            param.requires_grad_(False)

    def update(self, model):
        with torch.no_grad():
            for ema_param, param in zip(self.ema.parameters(), model.parameters()):
                ema_param.data.mul_(self.decay).add_(param.data, alpha=1 - self.decay)
            for ema_buffer, buffer in zip(self.ema.buffers(), model.buffers()):
                ema_buffer.copy_(buffer)


model = TransReID(
    num_classes=num_classes,
    num_cameras=num_cameras,
    embed_dim=768,
    vit_model=VIT_MODEL,
    pretrained=False,
    sie_camera=True,
    jpm=True,
    img_size=IMG_SIZE,
).to(DEVICE)

print(f"Parameters: {sum(param.numel() for param in model.parameters()):,}")

In [ ]:
# -- Load the original 09 CityFlowV2 checkpoint for extended fine-tuning --
print(f"Loading extended-training checkpoint from {RESUME_CHECKPOINT_PATH}")

checkpoint = torch.load(str(RESUME_CHECKPOINT_PATH), map_location="cpu", weights_only=False)
state_dict = checkpoint.get("state_dict", checkpoint) if isinstance(checkpoint, dict) else checkpoint
load_result = model.load_state_dict(state_dict, strict=True)
print(f"  Loaded tensors: {len(state_dict)}")
print(f"  Missing: {list(load_result.missing_keys)}")
print(f"  Unexpected: {list(load_result.unexpected_keys)}")
print("  Full TransReID state restored: backbone + BNNeck + heads + SIE/JPM")

if NUM_GPUS > 1:
    model = nn.DataParallel(model)
    print(f"  Wrapped in DataParallel ({NUM_GPUS} GPUs)")

## 6. Training

Extended fine-tuning strategy:
- **120 additional epochs** at 256x256 resolution
- **Lower LR**: backbone_lr=5e-5, head_lr=5e-4, LLRD=0.75
- **Fresh cosine schedule**: 5-epoch warmup + 110-step cosine decay
- **No backbone freeze**: resume directly from the fully trained 09 checkpoint
- **Losses**: CE(eps=0.05) + Circle(m=0.25, gamma=128) + Center(5e-4) from epoch 1
- **Original 09 augmentations**: flip, pad+crop, weaker jitter, normalized random erasing
- **Chunked evaluation** every 20 epochs for safer metric computation

In [ ]:
# -- Training config: extended fine-tuning from the original 09 best checkpoint --
try:
    from src.training.losses import CrossEntropyLabelSmooth, CircleLoss, CenterLoss
except ModuleNotFoundError as import_error:
    if not getattr(import_error, "name", "").startswith("src"):
        raise

    import torch.nn as nn
    import torch.nn.functional as F

    class CrossEntropyLabelSmooth(nn.Module):
        def __init__(self, num_classes, epsilon=0.1):
            super().__init__()
            self.num_classes = num_classes
            self.epsilon = epsilon

        def forward(self, inputs, targets):
            log_probs = F.log_softmax(inputs, dim=1)
            targets = F.one_hot(targets, num_classes=self.num_classes).float()
            targets = (1.0 - self.epsilon) * targets + self.epsilon / self.num_classes
            return (-targets * log_probs).sum(dim=1).mean()

    class CircleLoss(nn.Module):
        def __init__(self, m=0.25, gamma=128):
            super().__init__()
            self.m = m
            self.gamma = gamma

        def forward(self, feats, labels):
            if feats.ndim != 2:
                raise ValueError(f"Expected 2D features, got {tuple(feats.shape)}")

            feats = F.normalize(feats, p=2, dim=1)
            sim_mat = feats @ feats.t()
            labels = labels.view(-1, 1)
            same_identity = labels.eq(labels.t())
            positive_mask = torch.triu(same_identity, diagonal=1)
            negative_mask = torch.triu(~same_identity, diagonal=1)
            sp = sim_mat[positive_mask]
            sn = sim_mat[negative_mask]

            if sp.numel() == 0 or sn.numel() == 0:
                return feats.sum() * 0.0

            ap = torch.clamp_min(-sp.detach() + 1 + self.m, 0.0)
            an = torch.clamp_min(sn.detach() + self.m, 0.0)
            delta_p = 1 - self.m
            delta_n = self.m
            logit_p = -self.gamma * ap * (sp - delta_p)
            logit_n = self.gamma * an * (sn - delta_n)
            return F.softplus(torch.logsumexp(logit_n, dim=0) + torch.logsumexp(logit_p, dim=0))

    class CenterLoss(nn.Module):
        def __init__(self, num_classes, feat_dim):
            super().__init__()
            self.num_classes = num_classes
            self.feat_dim = feat_dim
            self.centers = nn.Parameter(torch.randn(num_classes, feat_dim))

        def forward(self, features, labels):
            labels = labels.long()
            batch_centers = self.centers.index_select(0, labels)
            return 0.5 * (features - batch_centers).pow(2).sum(dim=1).mean()

    print(f"src.training.losses unavailable ({import_error}); using in-notebook fallback losses")

ce_loss = CrossEntropyLabelSmooth(num_classes, 0.05).to(DEVICE)
circle_loss_fn = CircleLoss(m=0.25, gamma=128).to(DEVICE)
ctr_loss = CenterLoss(num_classes, 768).to(DEVICE)
CENTER_WEIGHT = 5e-4
CENTER_START = 0

raw_model = model.module if hasattr(model, "module") else model

backbone_lr = 5e-5
head_lr = 5e-4
wd = 5e-4
llrd_factor = 0.75

param_groups = (raw_model.module if isinstance(raw_model, torch.nn.DataParallel) else raw_model).get_llrd_param_groups(backbone_lr, head_lr, decay=llrd_factor)
optimizer = torch.optim.AdamW(param_groups, weight_decay=wd)
center_optimizer = torch.optim.SGD(ctr_loss.parameters(), lr=0.5)
base_lrs = [group["lr"] for group in optimizer.param_groups]

EPOCHS = 120
WARMUP = 5
T_MAX = 110

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=T_MAX)
scaler = torch.amp.GradScaler("cuda")

history = {
    "loss": [],
    "mAP": [],
    "R1": [],
    "mAP_rr": [],
    "R1_rr": [],
}
best_mAP = 0.0
best_model_path = BEST_MODEL_PATH_A

print("=" * 70)
print(f"  Extended fine-tuning TransReID on CityFlowV2 ({num_classes} IDs, {num_cameras} cameras)")
print(f"  Resume: {RESUME_CHECKPOINT_PATH}")
print("  Full state restore, fresh optimizer, fresh cosine schedule")
print(f"  Losses: CE(eps=0.05) + Circle(m=0.25, gamma=128) + Center(5e-4, start@ep{CENTER_START})")
print(f"  LR: backbone={backbone_lr}, head={head_lr}, LLRD={llrd_factor}, weight_decay={wd}")
print(f"  Epochs: {EPOCHS}, warmup: {WARMUP}, T_max: {T_MAX}, resolution: {H}x{W}")
print(f"  Eval chunk size: {EVAL_CHUNK_SIZE}")
print("=" * 70)
t0 = time.time()
for epoch in range(EPOCHS):
    model.train()
    rl, nb = 0.0, 0
    use_center = (epoch >= CENTER_START)

    if epoch < WARMUP:
        wf = (epoch + 1) / WARMUP
        for i, pg in enumerate(optimizer.param_groups):
            pg["lr"] = base_lrs[i] * wf
    elif epoch == WARMUP:
        for i, pg in enumerate(optimizer.param_groups):
            pg["lr"] = base_lrs[i]

    for imgs, pids, cams, _ in train_loader:
        imgs, pids, cams = imgs.to(DEVICE), pids.to(DEVICE).long(), cams.to(DEVICE).long()
        optimizer.zero_grad()
        if use_center:
            center_optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            out = model(imgs, cam_ids=cams)
            if len(out) == 3:
                c, f, jc = out
                loss = ce_loss(c, pids) + circle_loss_fn(f, pids) + 0.5 * ce_loss(jc, pids)
            else:
                c, f = out
                loss = ce_loss(c, pids) + circle_loss_fn(f, pids)

        if use_center:
            center_l = ctr_loss(f.float(), pids)
            total_loss = loss + CENTER_WEIGHT * center_l
            scaler.scale(total_loss).backward()
            scaler.unscale_(optimizer)
            scaler.unscale_(center_optimizer)
            torch.nn.utils.clip_grad_norm_(raw_model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.step(center_optimizer)
        else:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(raw_model.parameters(), max_norm=5.0)
            scaler.step(optimizer)

        scaler.update()
        rl += loss.item() if not torch.isnan(loss) else 0.0
        nb += 1

    if epoch >= WARMUP:
        scheduler.step()
    history["loss"].append(rl / max(nb, 1))

    if (epoch + 1) % 10 == 0:
        hd_lr = optimizer.param_groups[-1]["lr"]
        ctr_tag = " [+center]" if use_center else ""
        print(f"Epoch {epoch+1:3d} | Loss {rl/max(nb,1):.4f} | head_lr={hd_lr:.2e}{ctr_tag}")

    if (epoch + 1) % 20 == 0 or epoch == EPOCHS - 1:
        mAP, cmc, mAP_rr, cmc_rr = full_eval(model, query_loader, gallery_loader, DEVICE, pass_cams=True)

        history["mAP"].append(mAP)
        history["R1"].append(cmc[0])
        history["mAP_rr"].append(mAP_rr or 0)
        history["R1_rr"].append(cmc_rr[0] if cmc_rr is not None else 0)

        is_best = mAP > best_mAP
        if is_best:
            best_mAP = mAP
            torch.save(raw_model.state_dict(), best_model_path)

        print(f"Eval @ epoch {epoch+1:3d}:")
        print(f"  Base  mAP: {mAP:.4f}, R1: {cmc[0]:.4f}{' BEST' if is_best else ''}")
        if mAP_rr is not None:
            print(f"  Base  mAP(RR): {mAP_rr:.4f}, R1(RR): {cmc_rr[0]:.4f}")

elapsed = time.time() - t0
torch.save(raw_model.state_dict(), OUTPUT_DIR / "transreid_cityflowv2_last_expA_resume.pth")
best_mAP_exp_a = best_mAP
exp_a_export_path = EXPORT_DIR / "vehicle_transreid_09q_expA_resume.pth"
if best_model_path.exists():
    if best_model_path.resolve() != exp_a_export_path.resolve():
        import shutil
        shutil.copy2(best_model_path, exp_a_export_path)
else:
    print(f"Warning: best checkpoint not found at {best_model_path}")
print(f"\nExperiment A Done | Best mAP: {best_mAP_exp_a:.4f}")

In [ ]:
# -- Load VeRi-776 pretrained weights (domain adaptation) --
# Backbone + BNNeck transfer; classifier head re-initialized for CityFlowV2.
# pos_embed is skipped because VeRi's is 224-sized; we keep timm's resized CLIP version.
PRETRAINED_PATH = None

search_paths = [
    # Kaggle Models (attached via kernel-metadata.json model_sources)
    Path("/kaggle/input/models/mrkdagods/transreid-veri/other/default/1/transreid_veri_best.pth"),
    # Correct filename from gumfreddy/mtmc-weights dataset (CityFlowV2 checkpoint)
    Path("/kaggle/input/mtmc-weights/vehicle_transreid_vit_base_cityflowv2.pth"),
    Path("/tmp/mtmc-weights/vehicle_transreid_vit_base_cityflowv2.pth"),
    # Fallbacks
    Path("models/reid/vehicle_transreid_vit_base_veri776.pth"),
]

for p in search_paths:
    if p.exists():
        PRETRAINED_PATH = p
        break

if PRETRAINED_PATH is not None:
    print(f"Loading VeRi-776 pretrained weights from {PRETRAINED_PATH}")
    ckpt = torch.load(str(PRETRAINED_PATH), map_location="cpu")
    state = ckpt.get("state_dict", ckpt)

    # Skip classifier head, SIE, and pos_embed (shape mismatch: 224 vs 256)
    skip_keys = ["cls_head", "jpm_cls", "sie_embed", "pos_embed"]
    filtered = {}
    skipped = []
    for k, v in state.items():
        if any(sk in k for sk in skip_keys):
            skipped.append(k)
            continue
        filtered[k] = v

    missing, unexpected = model.load_state_dict(filtered, strict=False)
    print(f"  Loaded: {len(filtered)} params")
    print(f"  Skipped (re-init): {skipped}")
    print(f"  Missing (new): {missing}")
    print(f"  Backbone + BNNeck transferred from VeRi-776")
else:
    print("No VeRi-776 pretrained weights found -- training from CLIP init")
    print("For best results, attach NB08 output as Kaggle dataset")

if NUM_GPUS > 1:
    model = nn.DataParallel(model)
    print(f"  Wrapped in DataParallel ({NUM_GPUS} GPUs)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("TransReID Extended -- CityFlowV2 Fine-Tune from 09 Best Checkpoint", fontsize=14, fontweight="bold")

ax = axes[0]
ax.plot(history["loss"], "b-", alpha=0.7)
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.set_title("Training Loss")
ax.grid(True, alpha=0.3)

ax = axes[1]
ee = [20 * (i + 1) for i in range(len(history["mAP"]))]
if history["mAP"]:
    ax.plot(ee, [metric * 100 for metric in history["mAP"]], "r-o", label="mAP")
    ax.plot(ee, [metric * 100 for metric in history["R1"]], "b-s", label="R1")
    if any(history["mAP_rr"]):
        ax.plot(ee, [metric * 100 for metric in history["mAP_rr"]], "r--^", label="mAP(RR)")
ax.set_xlabel("Epoch"); ax.set_ylabel("%"); ax.set_title("Metrics")
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Re-initialize model for Experiment B (fresh CLIP init)
model = TransReID(
    num_classes=num_classes,
    num_cameras=num_cameras,
    embed_dim=768,
    vit_model=VIT_MODEL,
    pretrained=False,
    sie_camera=True,
    jpm=True,
    img_size=IMG_SIZE,
).to(DEVICE)

# -- Training config (Experiment B: CircleLoss ablation, no EMA) --
ce_loss = CrossEntropyLabelSmooth(num_classes, 0.05).to(DEVICE)  # reduced from 0.1 for sharper R1
circle_loss_fn = CircleLoss(m=0.25, gamma=128).to(DEVICE)
ctr_loss = CenterLoss(num_classes, 768).to(DEVICE)
CENTER_WEIGHT = 5e-4
CENTER_START = 15

raw_model = model.module if hasattr(model, 'module') else model

backbone_lr = 1e-4
head_lr = 1e-3
wd = 5e-4
llrd_factor = 0.75

param_groups = (raw_model.module if isinstance(raw_model, torch.nn.DataParallel) else raw_model).get_llrd_param_groups(backbone_lr, head_lr, decay=llrd_factor)
optimizer = torch.optim.AdamW(param_groups, weight_decay=wd)
center_optimizer = torch.optim.SGD(ctr_loss.parameters(), lr=0.5)
base_lrs = [pg["lr"] for pg in optimizer.param_groups]

EPOCHS = 120
WARMUP = 10

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP)
scaler = torch.amp.GradScaler("cuda")

history = {
    "loss": [],
    "mAP": [],
    "R1": [],
    "mAP_rr": [],
    "R1_rr": [],
}
best_mAP = 0.0
best_model_path = BEST_MODEL_PATH_B

print("=" * 70)
print(f"  Fine-tuning TransReID on CityFlowV2 ({num_classes} IDs, {num_cameras} cameras)")
print(f"  Init: {'VeRi-776 pretrained' if PRETRAINED_PATH else 'CLIP only'}")
print(f"  Losses: CE(eps=0.05) + Circle(m=0.25, gamma=128) + Center(5e-4, delayed@ep{CENTER_START})")
print(f"  LR: backbone={backbone_lr}, head={head_lr}, LLRD={llrd_factor}")
print(f"  Epochs: {EPOCHS}, warmup: {WARMUP}, resolution: {H}x{W}")
print("  EMA: disabled for Experiment B")
print("=" * 70)

t0 = time.time()
for epoch in range(EPOCHS):
    model.train()
    rl, nb = 0.0, 0
    use_center = (epoch >= CENTER_START)

    if epoch < WARMUP:
        wf = (epoch + 1) / WARMUP
        for i, pg in enumerate(optimizer.param_groups):
            pg["lr"] = base_lrs[i] * wf
    elif epoch == WARMUP:
        for i, pg in enumerate(optimizer.param_groups):
            pg["lr"] = base_lrs[i]

    for imgs, pids, cams, _ in train_loader:
        imgs, pids, cams = imgs.to(DEVICE), pids.to(DEVICE).long(), cams.to(DEVICE).long()
        optimizer.zero_grad()
        if use_center:
            center_optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            out = model(imgs, cam_ids=cams)
            if len(out) == 3:
                c, f, jc = out
                loss = ce_loss(c, pids) + circle_loss_fn(f, pids) + 0.5 * ce_loss(jc, pids)
            else:
                c, f = out
                loss = ce_loss(c, pids) + circle_loss_fn(f, pids)

        if use_center:
            center_l = ctr_loss(f.float(), pids)
            total_loss = loss + CENTER_WEIGHT * center_l
            scaler.scale(total_loss).backward()
            scaler.unscale_(optimizer)
            scaler.unscale_(center_optimizer)
            torch.nn.utils.clip_grad_norm_(raw_model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.step(center_optimizer)
        else:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(raw_model.parameters(), max_norm=5.0)
            scaler.step(optimizer)

        scaler.update()
        rl += loss.item() if not torch.isnan(loss) else 0.0
        nb += 1

    if epoch >= WARMUP:
        scheduler.step()
    history["loss"].append(rl / max(nb, 1))

    if (epoch + 1) % 10 == 0:
        hd_lr = optimizer.param_groups[-1]["lr"]
        ctr_tag = " [+center]" if use_center else ""
        print(f"Epoch {epoch+1:3d} | Loss {rl/max(nb,1):.4f} | head_lr={hd_lr:.2e}{ctr_tag}")

    if (epoch + 1) % 20 == 0 or epoch == EPOCHS - 1:
        mAP, cmc, mAP_rr, cmc_rr = full_eval(model, query_loader, gallery_loader, DEVICE, pass_cams=True)

        history["mAP"].append(mAP)
        history["R1"].append(cmc[0])
        history["mAP_rr"].append(mAP_rr or 0)
        history["R1_rr"].append(cmc_rr[0] if cmc_rr is not None else 0)

        is_best = mAP > best_mAP
        if is_best:
            best_mAP = mAP
            torch.save(raw_model.state_dict(), best_model_path)

        print(f"Eval @ epoch {epoch+1:3d}:")
        print(f"  Base  mAP: {mAP:.4f}, R1: {cmc[0]:.4f}{' BEST' if is_best else ''}")
        if mAP_rr is not None:
            print(f"  Base  mAP(RR): {mAP_rr:.4f}, R1(RR): {cmc_rr[0]:.4f}")

elapsed = time.time() - t0
torch.save(raw_model.state_dict(), OUTPUT_DIR / "transreid_cityflowv2_last.pth")
print(f"\nDone in {elapsed/3600:.1f}h | Best mAP: {best_mAP:.4f}")

# -- Export the extended base model --
best_state_path = best_model_path if best_model_path.exists() else LAST_MODEL_PATH

best_state = torch.load(best_state_path, map_location="cpu")
export_path = EXPORT_DIR / "vehicle_transreid_vit_base_cityflowv2_extended.pth"
torch.save({"state_dict": best_state}, export_path)
print(f"Model exported to {export_path} ({export_path.stat().st_size / 1e6:.1f} MB)")

metadata = {
    "task": "vehicle_reid",
    "dataset": "cityflowv2",
    "source_dataset": "AI City Challenge 2022 Track 1",
    "init_weights": "09_best_checkpoint",
    "variant": "v5_extended_resume",
    "experiment": "Extended fine-tuning from the original 09 80.14% mAP checkpoint",
    "training": f"TransReID ViT-Base CLIP CityFlowV2 resume ({EPOCHS}ep, lower LR, chunked eval)",
    "resume_checkpoint_path": str(RESUME_CHECKPOINT_PATH),
    "model": {
        "architecture": VIT_MODEL,
        "type": "transreid",
        "tricks": [
            "SIE",
            "JPM",
            "BNNeck",
            "CE+LS(0.05)",
            "CircleLoss(m=0.25,gamma=128)",
            "CenterLoss(start@ep0)",
            "CosLR",
            "RE",
            "CLIP-norm",
            "AugBaseline",
            f"LLRD({llrd_factor})",
            "ChunkedEval",
        ],
        "embedding_dim": 768,
        "input_size": [256, 256],
        "normalization": {"mean": CLIP_MEAN, "std": CLIP_STD},
        "num_cameras": num_cameras,
        "num_classes": num_classes,
        "best_mAP": float(best_mAP),
        "best_mAP_rr": float(max(history["mAP_rr"])) if history["mAP_rr"] else None,
        "epochs": EPOCHS,
        "backbone_lr": backbone_lr,
        "head_lr": head_lr,
        "center_loss_weight": CENTER_WEIGHT,
        "center_loss_start_epoch": CENTER_START,
        "label_smoothing": 0.05,
        "ema_enabled": False,
        "eval_chunk_size": EVAL_CHUNK_SIZE,
    },
    "exports": {
        "base": str(export_path),
        "best_checkpoint": str(best_state_path),
    },
    "cameras": CAMERAS,
    "split": {
        "train_images": len(train_data),
        "train_ids": num_classes,
        "query_images": len(query_data),
        "gallery_images": len(gallery_data),
        "eval_ids": len(eval_ids),
    },
}

meta_path = EXPORT_DIR / "vehicle_reid_cityflowv2_extended_metadata.json"
with open(meta_path, "w", encoding="utf-8") as handle:
    json.dump(metadata, handle, ensure_ascii=True, indent=2)
print(f"Metadata saved to {meta_path}")

print("\n" + "=" * 60)
print("RESULTS SUMMARY -- CityFlowV2 Vehicle ReID Extended Fine-Tune")
print("=" * 60)
print(f"Dataset:    CityFlowV2 ({len(CAMERAS)} cameras)")
print(f"Train:      {len(train_data)} images, {num_classes} IDs")
print(f"Eval:       {len(query_data)} query + {len(gallery_data)} gallery")
print("Model:      TransReID ViT-Base/16 CLIP")
print(f"Resume:     {RESUME_CHECKPOINT_PATH}")
print(f"Epochs:     {EPOCHS}")
if history["mAP"]:
    print(f"Best mAP:   {best_mAP * 100:.2f}%")
    print(f"Best R1:    {max(history['R1']) * 100:.2f}%")
print(f"Best ckpt:  {best_state_path}")
print("=" * 60)

## 9. Inference Integration

### Using the extended model in the MTMC pipeline

After training, download the exported model and metadata:

```bash
cp vehicle_transreid_vit_base_cityflowv2_extended.pth  models/reid/
cp vehicle_reid_cityflowv2_extended_metadata.json      models/reid/
```

### Update config to use the extended CityFlowV2-trained model

In `configs/default.yaml` or `configs/datasets/cityflowv2.yaml`:

```yaml
stage2:
  reid:
    vehicle:
      model_name: "transreid"
      weights_path: "models/reid/vehicle_transreid_vit_base_cityflowv2_extended.pth"
      embedding_dim: 768
      input_size: [256, 256]
      vit_model: "vit_base_patch16_clip_224.openai"
      num_cameras: 6
      clip_normalization: true
```

### Run full pipeline with MTMC evaluation

```bash
python scripts/run_pipeline.py --config configs/datasets/cityflowv2.yaml
```

## 8. Export Model

In [ ]:
# -- Export Experiment A and B models --
a_path = BEST_MODEL_PATH_A if BEST_MODEL_PATH_A.exists() else BEST_MODEL_PATH
b_path = BEST_MODEL_PATH_B if BEST_MODEL_PATH_B.exists() else None

if a_path.exists():
    export_a = EXPORT_DIR / "vehicle_transreid_09q_expA_extended.pth"
    torch.save(torch.load(a_path, map_location="cpu"), export_a)
    print(f"Experiment A exported to {export_a} ({export_a.stat().st_size/1e6:.1f} MB)")

if b_path and b_path.exists():
    export_b = EXPORT_DIR / "vehicle_transreid_09q_expB_scratch.pth"
    torch.save(torch.load(b_path, map_location="cpu"), export_b)
    print(f"Experiment B exported to {export_b} ({export_b.stat().st_size/1e6:.1f} MB)")

# Legacy export path (keep for compatibility)
best_state_path = a_path if a_path.exists() else (b_path if b_path else BEST_MODEL_PATH)
export_path = EXPORT_DIR / "vehicle_transreid_vit_base_cityflowv2.pth"
torch.save(torch.load(str(best_state_path), map_location="cpu"), export_path)
print(f"Primary export to {export_path} ({export_path.stat().st_size/1e6:.1f} MB)")

metadata = {
    "task": "vehicle_reid",
    "dataset": "cityflowv2",
    "source_dataset": "AI City Challenge 2022 Track 1",
    "variant": "09q_dual_experiment",
    "model": {
        "architecture": VIT_MODEL,
        "embedding_dim": 768,
        "input_size": [256, 256],
        "normalization": {"mean": CLIP_MEAN, "std": CLIP_STD},
        "num_cameras": num_cameras,
        "num_classes": num_classes,
    },
    "cameras": CAMERAS,
    "split": {
        "train_images": len(train_data),
        "train_ids": num_classes,
        "query_images": len(query_data),
        "gallery_images": len(gallery_data),
        "eval_ids": len(eval_ids),
    },
}

meta_path = EXPORT_DIR / "vehicle_reid_cityflowv2_metadata.json"
with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"Metadata saved to {meta_path}")

## 9. Inference Integration

### Using the trained model in the MTMC pipeline

After training, download the exported model and metadata:

```bash
# Copy exported files to local project
cp vehicle_transreid_vit_base_cityflowv2.pth  models/reid/
cp vehicle_reid_cityflowv2_metadata.json       models/reid/
```

### Update config to use CityFlowV2-trained model

In `configs/default.yaml` or `configs/datasets/cityflowv2.yaml`:

```yaml
stage2:
  reid:
    vehicle:
      model_name: "transreid"
      weights_path: "models/reid/vehicle_transreid_vit_base_cityflowv2.pth"
      embedding_dim: 768
      input_size: [256, 256]
      vit_model: "vit_base_patch16_clip_224.openai"
      num_cameras: 6
      clip_normalization: true
```

### Run full pipeline with MTMC evaluation

```bash
# Run full pipeline on CityFlowV2
python scripts/run_pipeline.py --config configs/datasets/cityflowv2.yaml

# Or evaluate ReID only
python scripts/eval_cityflowv2_reid.py \
    --weights models/reid/vehicle_transreid_vit_base_cityflowv2.pth
```

### Expected metrics
| Metric | ReID (Stage 2) | MTMC (Stage 5) |
|--------|---------------|----------------|
| mAP | 40-65% | -- |
| Rank-1 | 55-80% | -- |
| IDF1 | -- | 70-84% |
| HOTA | -- | 50-65% |
| MOTA | -- | 60-78% |